# UltraGPT: Production-Grade Decoder-Only Transformer from Scratch
### End-to-End Implementation: Configuration, Scraping & ETL, Tokenization, Architecture, Training Engine, Context Bot, and OpenAI Serving Layer

This notebook is an exact, self-contained, phase-by-phase mirror of the UltraGPT codebase. Every phase contains the exact, unabridged production implementation with complete type hints and docstrings.

---

### Roadmap:
- **Phase 1**: System Verification & Hardware Telemetry (GPU / CUDA Detection)
- **Phase 2**: Centralized Configuration & Presets (`GPTConfig`)
- **Phase 3**: Data Ingestion, Web Scraping & Text Cleaning Pipeline
- **Phase 4**: Tokenization Engine (BPE & Character-Level with Specials)
- **Phase 5**: PyTorch Dataset & DataLoader (Causal LM Label Shifting)
- **Phase 6**: Decoder-Only GPT Architecture from Scratch (Attention, MLP, Block, GPT)
- **Phase 7**: Training Engine (Decoupled AdamW, Cosine Warmup, AMP & Checkpoint Manager)
- **Phase 8**: Model Training Execution & Telemetry Demonstration
- **Phase 9**: Post-Training Loss Curves & LR Schedule Visualization
- **Phase 10**: Context-Aware Conversational Bot (Sampling & Memory)
- **Phase 11**: OpenAI-Compatible FastAPI Serving Layer & Live Verification


## Phase 1: Environment Setup & Hardware Telemetry
We inspect the computing environment, checking CUDA GPU device properties, driver compatibility, and memory allocation.


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import math
import time
import json
import logging
from pathlib import Path
from typing import Any, Dict, Generator, List, Literal, Optional, Sequence, Set, Tuple, Union

# Ensure current project directory is in python search path
if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Check PyTorch & GPU Acceleration
print(f"PyTorch Version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available (NVIDIA GPU): {cuda_available}")

if cuda_available:
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    print("Active Device: Apple Silicon MPS (Metal Performance Shaders)")
else:
    device = "cpu"
    print("Active Device: CPU")


## Phase 2: Centralized Configuration & Presets (`GPTConfig`)
We load the centralized dataclass configuration governing all architectural dimensions, presets (`gpt-nano`, `gpt-micro`, `gpt-mini`, `gpt2-small`), training parameters, and device resolution.


In [ ]:
"""
Centralized Configuration & Hyperparameter Definitions for Decoder-Only GPT.
Production-grade dataclass supporting presets, user overrides, and serialization.
"""

from __future__ import annotations

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Optional


@dataclass
class GPTConfig:
    # ---------------------------------------------------------
    # 1. Model Architecture Dimensions
    # ---------------------------------------------------------
    vocab_size: int = 50257       # Vocabulary size (default: GPT-2 BPE size)
    n_embd: int = 768             # Embedding dimension / hidden size
    n_layer: int = 12             # Number of Transformer blocks
    n_head: int = 12              # Number of attention heads (n_embd must be divisible by n_head)
    block_size: int = 1024        # Maximum context length (sequence length)
    dropout: float = 0.0          # Dropout probability across embeddings, attention & MLP
    bias: bool = True             # Enable bias in Linears and LayerNorms (like GPT-2)

    # ---------------------------------------------------------
    # 2. Training Hyperparameters
    # ---------------------------------------------------------
    batch_size: int = 64          # Micro-batch size per optimization step
    epochs: Optional[int] = None  # Number of complete passes over the dataset (optional)
    learning_rate: float = 6e-4   # Peak learning rate
    min_lr: float = 6e-5          # Minimum learning rate for cosine decay schedule
    warmup_iters: int = 200       # Iterations of linear warmup
    lr_decay_iters: int = 2000    # Total iterations over which learning rate decays
    max_iters: int = 2000         # Total training steps
    eval_interval: int = 100      # Validate model every N iterations
    eval_iters: int = 20          # Number of batches sampled during evaluation
    grad_clip: float = 1.0        # Maximum gradient norm for clipping
    weight_decay: float = 0.1     # Decoupled weight decay for 2D weight tensors
    mixed_precision: str = "fp16" # 'bf16', 'fp16', or 'no'

    # ---------------------------------------------------------
    # 3. System, Device & Storage
    # ---------------------------------------------------------
    device: str = "auto"          # 'auto', 'cuda', 'mps', or 'cpu'
    seed: int = 1337              # Deterministic random seed
    checkpoint_dir: str = "checkpoints" # Directory for rolling & versioned checkpoints
    save_checkpoint_interval: int = 500 # Interval for versioned snapshot creation
    dataset_path: str = "data/dataset.txt" # Path to primary raw/cleaned dataset
    preset_name: Optional[str] = None # Name of preset if loaded from one

    def __post_init__(self) -> None:
        """Validate architectural integrity and resolve runtime device."""
        if self.n_embd % self.n_head != 0:
            raise ValueError(
                f"Embedding dimension n_embd={self.n_embd} must be divisible by n_head={self.n_head}"
            )
        if self.device == "auto":
            self.device = self._detect_device()

    @staticmethod
    def _detect_device() -> str:
        """Auto-detect available acceleration backend: CUDA -> MPS -> CPU."""
        try:
            import torch
            if torch.cuda.is_available():
                return "cuda"
            if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
                return "mps"
        except ImportError:
            pass
        return "cpu"

    @classmethod
    def get_presets(cls) -> Dict[str, Dict[str, Any]]:
        """
        Standard architecture presets:
          - gpt-nano:  ~1.0M parameters (rapid CPU prototyping / unit tests)
          - gpt-micro: ~10.5M parameters (single consumer GPU quick training)
          - gpt-mini:  ~25.8M parameters (solid desktop / edge training)
          - gpt2-small: ~124M parameters (standard OpenAI GPT-2 architecture)
        """
        return {
            "gpt-nano": {
                "n_layer": 3,
                "n_head": 3,
                "n_embd": 48,
                "block_size": 128,
                "vocab_size": 50257,
                "dropout": 0.0,
                "bias": True,
                "batch_size": 16,
                "learning_rate": 1e-3,
                "min_lr": 1e-4,
                "warmup_iters": 50,
                "lr_decay_iters": 1000,
                "max_iters": 1000,
            },
            "gpt-micro": {
                "n_layer": 4,
                "n_head": 4,
                "n_embd": 128,
                "block_size": 256,
                "vocab_size": 50257,
                "dropout": 0.1,
                "bias": True,
                "batch_size": 32,
                "learning_rate": 8e-4,
                "min_lr": 8e-5,
                "warmup_iters": 100,
                "lr_decay_iters": 2000,
                "max_iters": 2000,
            },
            "gpt-mini": {
                "n_layer": 6,
                "n_head": 6,
                "n_embd": 192,
                "block_size": 512,
                "vocab_size": 50257,
                "dropout": 0.1,
                "bias": True,
                "batch_size": 32,
                "learning_rate": 6e-4,
                "min_lr": 6e-5,
                "warmup_iters": 200,
                "lr_decay_iters": 3000,
                "max_iters": 3000,
            },
            "gpt2-small": {
                "n_layer": 12,
                "n_head": 12,
                "n_embd": 768,
                "block_size": 1024,
                "vocab_size": 50257,
                "dropout": 0.1,
                "bias": True,
                "batch_size": 64,
                "learning_rate": 6e-4,
                "min_lr": 6e-5,
                "warmup_iters": 200,
                "lr_decay_iters": 5000,
                "max_iters": 5000,
            },
        }

    @classmethod
    def from_preset(cls, preset_name: str, **overrides: Any) -> GPTConfig:
        """Instantiate GPTConfig using a designated preset with custom parameter overrides."""
        presets = cls.get_presets()
        if preset_name not in presets:
            raise KeyError(
                f"Unknown preset '{preset_name}'. Available: {list(presets.keys())}"
            )
        preset_args = presets[preset_name].copy()
        preset_args.update(overrides)
        preset_args["preset_name"] = preset_name
        return cls(**preset_args)

    def to_dict(self) -> Dict[str, Any]:
        """Convert configuration to dictionary."""
        return asdict(self)

    def save_json(self, filepath: str | Path) -> None:
        """Serialize configuration to a JSON file."""
        path = Path(filepath)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def load_json(cls, filepath: str | Path) -> GPTConfig:
        """Load configuration from a JSON file."""
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        return cls(**data)


# Verify presets and parameter counts
for preset_name in ["gpt-nano", "gpt-micro", "gpt-mini", "gpt2-small"]:
    cfg_inst = GPTConfig.from_preset(preset_name)
    print(f"Preset [{preset_name:10s}]: layers={cfg_inst.n_layer:2d}, heads={cfg_inst.n_head:2d}, "
          f"embd={cfg_inst.n_embd:4d}, context={cfg_inst.block_size:4d}, batch={cfg_inst.batch_size:2d}")


## Phase 3: Data Ingestion, Web Scraping & Text Cleaning Pipeline
Here we define:
1. `TextCleaner`: Unicode NFKC normalization, regex citation removal (`[1]`, `[citation needed]`, `[edit]`), HTML tag stripping, and whitespace consolidation.
2. `DialogueFormatter`: ShareGPT / ChatML format converter (`<|im_start|>role...<|im_end|>`).
3. `WebScraper`: BeautifulSoup + requests crawler targeting Wikipedia and web articles.
Then we run the scraper to ingest real articles into `data/dataset.txt`.


In [ ]:
"""
Text Cleaning Engine and Dialogue Format Converter.
Handles NFKC normalization, artifact removal, HTML strip, and MLChat/ShareGPT formatting.
"""

from __future__ import annotations

import html
import re
import unicodedata
from typing import Dict, List, Optional


class TextCleaner:
    """
    Production-grade text cleaning engine:
    1. Unicode Normalization (NFKC).
    2. HTML tag and entity stripping.
    3. Citation/reference artifact removal ([1], [citation needed], [edit], etc.).
    4. Redundant whitespace consolidation and non-printable char filtering.
    """

    # Compiled regex patterns for high-throughput cleaning
    CITATION_PATTERN = re.compile(r"\[(?:\d+|citation needed|edit|note \d+)\]", re.IGNORECASE)
    HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
    URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
    NON_PRINTABLE_PATTERN = re.compile(r"[^\x20-\x7E\n\r\t\u00A0-\uFFFF]")
    MULTI_NEWLINE_PATTERN = re.compile(r"\n{3,}")
    MULTI_SPACE_PATTERN = re.compile(r"[ \t]{2,}")

    @classmethod
    def clean(
        cls,
        text: str,
        remove_urls: bool = False,
        remove_citations: bool = True,
        strip_html: bool = True,
    ) -> str:
        """Execute full cleaning pipeline on input string."""
        if not text:
            return ""

        # Step 1: Unicode NFKC Normalization
        text = unicodedata.normalize("NFKC", text)

        # Step 2: Unescape HTML entities & strip HTML residue
        if strip_html:
            text = html.unescape(text)
            text = cls.HTML_TAG_PATTERN.sub(" ", text)

        # Step 3: Remove citation artifacts (e.g. "[1]", "[edit]")
        if remove_citations:
            text = cls.CITATION_PATTERN.sub("", text)

        # Step 4: Optional URL removal
        if remove_urls:
            text = cls.URL_PATTERN.sub("", text)

        # Step 5: Filter non-printable control characters (preserving \n and \t)
        text = cls.NON_PRINTABLE_PATTERN.sub("", text)

        # Step 6: Collapse whitespace while preserving structural paragraph breaks
        lines = [cls.MULTI_SPACE_PATTERN.sub(" ", line).strip() for line in text.splitlines()]
        text = "\n".join(lines)
        text = cls.MULTI_NEWLINE_PATTERN.sub("\n\n", text)

        return text.strip()


class DialogueFormatter:
    """
    Format converter for multi-turn structured dialogues:
    MLChat / ShareGPT / ChatML style:
    <|im_start|>system
    {system_message}<|im_end|>
    <|im_start|>user
    {user_message}<|im_end|>
    <|im_start|>assistant
    {assistant_message}<|im_end|>
    """

    IM_START = "<|im_start|>"
    IM_END = "<|im_end|>"

    @classmethod
    def format_turn(cls, role: str, content: str) -> str:
        """Format a single dialogue turn."""
        clean_content = content.strip()
        return f"{cls.IM_START}{role}\n{clean_content}{cls.IM_END}"

    @classmethod
    def format_conversation(
        cls,
        messages: List[Dict[str, str]],
        add_generation_prompt: bool = False,
    ) -> str:
        """
        Convert list of message dicts ({'role': 'user'|'assistant'|'system', 'content': '...'})
        into a formatted single prompt string.
        """
        formatted = []
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            formatted.append(cls.format_turn(role, content))

        result = "\n".join(formatted)
        if add_generation_prompt:
            result += f"\n{cls.IM_START}assistant\n"
        return result

    @classmethod
    def parse_conversation(cls, text: str) -> List[Dict[str, str]]:
        """Parse structured ChatML string back into a list of message dicts."""
        pattern = re.compile(
            rf"{re.escape(cls.IM_START)}(system|user|assistant)\n(.*?){re.escape(cls.IM_END)}",
            re.DOTALL,
        )
        matches = pattern.findall(text)
        return [{"role": role, "content": content.strip()} for role, content in matches]


In [ ]:
"""
Lightweight Web Scraper for Dataset Ingestion.
Crawls textual content from Wikipedia (exact titles or search keywords),
general web articles, and raw text URLs, runs through TextCleaner,
and saves/appends to dataset storage.
"""

from __future__ import annotations

import logging
from pathlib import Path
from typing import List, Optional
import urllib.parse

import requests
from bs4 import BeautifulSoup


logger = logging.getLogger(__name__)


class WebScraper:
    """
    HTTP Crawler & Text Ingestion Utility:
    - Search Wikipedia articles by any keyword, phrase, or topic.
    - Scrape general web URLs.
    - Automatic DOM noise filtering and text sanitization.
    - Append to dataset.txt with <|endoftext|> boundaries.
    """

    DEFAULT_USER_AGENT = (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36 UltraGPT/1.0"
    )

    def __init__(self, output_path: str | Path = "data/dataset.txt", timeout: int = 15):
        self.output_path = Path(output_path)
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": self.DEFAULT_USER_AGENT})

    def search_wikipedia(self, query: str) -> Optional[str]:
        """
        Search Wikipedia using OpenSearch API for a keyword/phrase.
        Returns the top matching article URL, or None if no match found.
        """
        api_url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "opensearch",
            "search": query,
            "limit": 1,
            "namespace": 0,
            "format": "json",
        }
        try:
            resp = self.session.get(api_url, params=params, timeout=self.timeout)
            if resp.status_code == 200:
                data = resp.json()
                # data format: [query, [titles], [descriptions], [urls]]
                if len(data) >= 4 and len(data[3]) > 0:
                    matched_url = data[3][0]
                    matched_title = data[1][0]
                    print(f"  -> Search '{query}' matched Wikipedia article: '{matched_title}' ({matched_url})")
                    return matched_url
        except Exception as e:
            logger.warning(f"Wikipedia search query failed for '{query}': {e}")
        return None

    def scrape_url(self, url: str) -> str:
        """Fetch raw HTML, extract body text, and clean it."""
        try:
            response = self.session.get(url, timeout=self.timeout)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            # Remove scripts, styles, navigation, footer elements
            for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
                tag.decompose()

            # Target Wikipedia main content or generic article body
            content_div = soup.find("div", {"id": "mw-content-text"}) or soup.find("article") or soup.body
            if not content_div:
                return ""

            paragraphs = content_div.find_all("p")
            raw_text = "\n\n".join(p.get_text() for p in paragraphs if p.get_text().strip())
            cleaned_text = TextCleaner.clean(raw_text)
            return cleaned_text
        except Exception as e:
            logger.warning(f"Failed to scrape {url}: {e}")
            return ""

    def scrape_wikipedia(self, title_or_query: str) -> str:
        """
        Scrape Wikipedia by title, query phrase, or direct URL.
        If not a URL, it queries Wikipedia search to find the most relevant article.
        """
        if title_or_query.startswith("http://") or title_or_query.startswith("https://"):
            return self.scrape_url(title_or_query)

        # First try search API for best match
        matched_url = self.search_wikipedia(title_or_query)
        if matched_url:
            return self.scrape_url(matched_url)

        # Fallback: direct title encoding
        encoded_title = urllib.parse.quote(title_or_query.replace(" ", "_"))
        direct_url = f"https://en.wikipedia.org/wiki/{encoded_title}"
        return self.scrape_url(direct_url)

    def append_to_dataset(self, text: str) -> int:
        """Append cleaned text to dataset file. Returns number of characters written."""
        if not text:
            return 0
        with open(self.output_path, "a", encoding="utf-8") as f:
            f.write(text + "\n\n<|endoftext|>\n\n")
        return len(text)

    def scrape_batch(self, sources: List[str]) -> int:
        """
        Scrape a list of search queries, Wikipedia topics, or direct URLs,
        and append all cleaned text to the dataset file.
        Returns total characters appended.
        """
        total_chars = 0
        for src in sources:
            src_clean = src.strip()
            if not src_clean:
                continue
            print(f"Ingesting: '{src_clean}'...")
            text = self.scrape_wikipedia(src_clean)
            chars = self.append_to_dataset(text)
            total_chars += chars
            print(f"  [+] Wrote {chars:,} characters from '{src_clean}'")
        return total_chars


# Execute Live Web Scraper: Ingest real articles into data/dataset.txt
scraper = WebScraper(output_path="data/dataset.txt")
topics = [
    "Artificial intelligence",
    "Transformer (deep learning architecture)",
    "Large language model",
]
print("Scraping educational corpus from Wikipedia...")
total_chars = scraper.scrape_batch(topics)
print(f"Total scraped & cleaned characters in dataset.txt: {total_chars:,}")


## Phase 4: Tokenization Engine (BPE & Character-Level with Specials)
We implement two tokenizers:
- `BPETokenizer`: High-speed Byte-Pair Encoding (`tiktoken`) with special tokens (`<|endoftext|>`, `<|pad|>`, `<|im_start|>`, `<|im_end|>`).
- `CharTokenizer`: Self-contained character-level tokenizer with exact mapping and JSON serialization.


In [ ]:
"""
Tokenizer Implementations: Byte-Pair Encoding (BPE) & Character-Level Fallback.
Both support special token handling (<|endoftext|>, <|pad|>, <|im_start|>, <|im_end|>).
"""

from __future__ import annotations

import json
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Dict, List, Optional, Set, Union


class BaseTokenizer(ABC):
    """Abstract base class for all tokenizers."""

    @property
    @abstractmethod
    def vocab_size(self) -> int:
        pass

    @property
    @abstractmethod
    def eos_token_id(self) -> int:
        pass

    @property
    @abstractmethod
    def pad_token_id(self) -> int:
        pass

    @abstractmethod
    def encode(self, text: str) -> List[int]:
        pass

    @abstractmethod
    def decode(self, tokens: List[int]) -> str:
        pass


class BPETokenizer(BaseTokenizer):
    """
    Byte-Pair Encoding (BPE) Tokenizer wrapping GPT-2's vocabulary
    via tiktoken with explicit special tokens (<|endoftext|>, <|pad|>, <|im_start|>, <|im_end|>).
    """

    SPECIAL_TOKENS = {
        "<|endoftext|>": 50256,
        "<|pad|>": 50257,
        "<|im_start|>": 50258,
        "<|im_end|>": 50259,
    }

    def __init__(self) -> None:
        try:
            import tiktoken
            self._enc = tiktoken.get_encoding("gpt2")
        except ImportError:
            raise ImportError("tiktoken is required for BPETokenizer. Install via `pip install tiktoken`.")

        self._special_tokens = self.SPECIAL_TOKENS.copy()
        self._reverse_special_tokens = {v: k for k, v in self._special_tokens.items()}
        # Base gpt2 vocab has 50257 tokens (0..50256 where 50256 is <|endoftext|>)
        # With added pad, im_start, im_end, vocab_size becomes 50260
        self._vocab_size = 50260

    @property
    def vocab_size(self) -> int:
        return self._vocab_size

    @property
    def eos_token_id(self) -> int:
        return self._special_tokens["<|endoftext|>"]

    @property
    def pad_token_id(self) -> int:
        return self._special_tokens["<|pad|>"]

    @property
    def im_start_token_id(self) -> int:
        return self._special_tokens["<|im_start|>"]

    @property
    def im_end_token_id(self) -> int:
        return self._special_tokens["<|im_end|>"]

    def encode(self, text: str, allowed_special: Union[str, Set[str]] = "all") -> List[int]:
        """Encode text to token IDs handling standard and special tokens."""
        if allowed_special == "all":
            allowed = set(self._special_tokens.keys())
        elif isinstance(allowed_special, set):
            allowed = allowed_special
        else:
            allowed = set()

        return self._enc.encode(text, allowed_special=allowed)

    def decode(self, tokens: List[int]) -> str:
        """Decode token IDs back to string."""
        return self._enc.decode(tokens)


class CharTokenizer(BaseTokenizer):
    """
    Self-contained Character-level Tokenizer with special token handling.
    Ideal for testing, zero-dependency deployment, and fast CPU training.
    """

    DEFAULT_SPECIALS = ["<|endoftext|>", "<|pad|>", "<|im_start|>", "<|im_end|>"]

    def __init__(self, chars: Optional[List[str]] = None):
        self.specials = self.DEFAULT_SPECIALS.copy()
        self.stoi: Dict[str, int] = {}
        self.itos: Dict[int, str] = {}

        if chars:
            self.build_vocab(chars)

    def build_vocab(self, chars: Union[str, List[str]]) -> None:
        """Construct character-to-index and index-to-character tables."""
        unique_chars = sorted(list(set(chars)))
        # Filter out characters that might clash with special token brackets
        all_symbols = self.specials + unique_chars
        self.stoi = {ch: i for i, ch in enumerate(all_symbols)}
        self.itos = {i: ch for i, ch in enumerate(all_symbols)}

    @property
    def vocab_size(self) -> int:
        return len(self.stoi) if self.stoi else len(self.specials)

    @property
    def eos_token_id(self) -> int:
        return self.stoi.get("<|endoftext|>", 0)

    @property
    def pad_token_id(self) -> int:
        return self.stoi.get("<|pad|>", 1)

    @property
    def im_start_token_id(self) -> int:
        return self.stoi.get("<|im_start|>", 2)

    @property
    def im_end_token_id(self) -> int:
        return self.stoi.get("<|im_end|>", 3)

    def encode(self, text: str) -> List[int]:
        """Encode text, recognizing special tokens or falling back to char indices."""
        tokens: List[int] = []
        i = 0
        n = len(text)
        while i < n:
            matched_special = False
            for sp in self.specials:
                if text.startswith(sp, i):
                    tokens.append(self.stoi[sp])
                    i += len(sp)
                    matched_special = True
                    break
            if not matched_special:
                ch = text[i]
                if ch in self.stoi:
                    tokens.append(self.stoi[ch])
                else:
                    # Map unknown char to pad or skip
                    tokens.append(self.pad_token_id)
                i += 1
        return tokens

    def decode(self, tokens: List[int]) -> str:
        """Decode token IDs to string."""
        return "".join([self.itos.get(idx, "") for idx in tokens])

    def save(self, filepath: str | Path) -> None:
        """Persist tokenizer vocabulary to JSON."""
        path = Path(filepath)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"stoi": self.stoi, "specials": self.specials}, f, indent=2)

    @classmethod
    def load(cls, filepath: str | Path) -> CharTokenizer:
        """Load tokenizer vocabulary from JSON."""
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.stoi = data["stoi"]
        tok.itos = {int(k): v for v, k in tok.stoi.items()}
        tok.specials = data["specials"]
        return tok


# Test Tokenizers
with open("data/dataset.txt", "r", encoding="utf-8") as f:
    raw_corpus = f.read()

# Build Character Tokenizer from scraped dataset
char_tok = CharTokenizer(list(raw_corpus))
char_tok.save("checkpoints/char_vocab.json")
print(f"CharTokenizer initialized from scraped data | Vocab Size: {char_tok.vocab_size}")

test_str = "UltraGPT learns language via self-attention. <|endoftext|>"
enc = char_tok.encode(test_str)
dec = char_tok.decode(enc)
print(f"Original: '{test_str}'")
print(f"Encoded Token IDs (first 15): {enc[:15]}...")
print(f"Decoded String: '{dec}'")


## Phase 5: PyTorch Dataset & DataLoader Pipeline (Causal LM Shifting)
We implement `GPTDataset` which creates causally shifted input-target pairs ($x = 	ext{tokens}[:-1]$, $y = 	ext{tokens}[1:]$), performs a 90/10 train/validation split, and configures dynamic memory pinning.


In [ ]:
"""
PyTorch Dataset and DataLoader Pipeline for Causal Language Modeling.
Features dynamic pinning, randomized context sampling, causal label shifting (x=tokens[:-1], y=tokens[1:]),
and automated 90/10 train/validation splits.
"""

from __future__ import annotations

from typing import Tuple, Union, Sequence
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDataset(Dataset):
    """
    Dataset for Causal Language Modeling.
    Slices input tokens into chunks of length (block_size + 1) to yield shifted (x, y) pairs:
    x = chunk[:-1]
    y = chunk[1:]
    """

    def __init__(
        self,
        tokens: Union[torch.Tensor, Sequence[int]],
        block_size: int,
    ) -> None:
        if isinstance(tokens, torch.Tensor):
            self.tokens = tokens.long()
        else:
            self.tokens = torch.tensor(tokens, dtype=torch.long)

        self.block_size = block_size
        if len(self.tokens) <= self.block_size:
            raise ValueError(
                f"Token sequence length ({len(self.tokens)}) must be greater than block_size ({self.block_size})."
            )

        # Number of non-overlapping or offset samples available
        self.total_samples = len(self.tokens) - self.block_size

    def __len__(self) -> int:
        return self.total_samples

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        # Extract chunk of block_size + 1 tokens
        chunk = self.tokens[idx : idx + self.block_size + 1]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y


def create_dataloaders(
    tokens: Union[torch.Tensor, Sequence[int]],
    block_size: int,
    batch_size: int,
    split_ratio: float = 0.9,
    pin_memory: bool = True,
    num_workers: int = 0,
) -> Tuple[DataLoader, DataLoader, GPTDataset, GPTDataset]:
    """
    Splits tokens into 90/10 train/val splits and constructs PyTorch DataLoaders
    with dynamic pinning and memory efficiency.
    """
    if not isinstance(tokens, torch.Tensor):
        tokens_tensor = torch.tensor(tokens, dtype=torch.long)
    else:
        tokens_tensor = tokens.long()

    n = len(tokens_tensor)
    split_idx = int(n * split_ratio)

    train_tokens = tokens_tensor[:split_idx]
    val_tokens = tokens_tensor[split_idx:]

    train_dataset = GPTDataset(train_tokens, block_size=block_size)
    val_dataset = GPTDataset(val_tokens, block_size=block_size)

    # Dynamic pinning flag: only pin memory if CUDA is available and requested
    use_pin_memory = pin_memory and torch.cuda.is_available()

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=use_pin_memory,
        num_workers=num_workers,
        drop_last=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=use_pin_memory,
        num_workers=num_workers,
        drop_last=False,
    )

    return train_loader, val_loader, train_dataset, val_dataset


# Encode scraped corpus into token sequence
tokens = char_tok.encode(raw_corpus)
print(f"Total tokens in scraped dataset: {len(tokens):,}")

# Create 90/10 Train and Validation DataLoaders
train_loader, val_loader, train_ds, val_ds = create_dataloaders(
    tokens=tokens,
    block_size=64,
    batch_size=8,
    split_ratio=0.9,
)

print(f"Train samples: {len(train_ds):,} | Validation samples: {len(val_ds):,}")
x_sample, y_sample = next(iter(train_loader))
print(f"Batch x shape: {x_sample.shape} | Batch y shape: {y_sample.shape}")
print(f"Causal Shift Validated (x[0, 1:] == y[0, :-1]): {torch.equal(x_sample[0, 1:], y_sample[0, :-1])}")


## Phase 6: Decoder-Only GPT Architecture from Scratch
Here we construct the neural network using raw `torch.nn.Module`:
1. `CausalSelfAttention`: Batched $Q,K,V$, FlashAttention (`F.scaled_dot_product_attention`) with manual causal mask fallback.
2. `MLP`: Position-wise feed-forward with GELU(approx="tanh").
3. `TransformerBlock`: Pre-LayerNorm architecture.
4. `GPT`: Token & positional embeddings, weight tying (`lm_head.weight = wte.weight`), GPT-2 weight initialization, and parameter counter.


In [ ]:
"""
Multi-Head Causal Self-Attention Module with FlashAttention & Manual Fallback.
"""

from __future__ import annotations

import math
import torch
import torch.nn as nn
import torch.nn.functional as F



class CausalSelfAttention(nn.Module):
    """
    Multi-Head Causal Self-Attention mechanism:
    - Batched Q, K, V linear projections.
    - Scaled dot-product attention utilizing torch.nn.functional.scaled_dot_product_attention
      (FlashAttention / memory-efficient attention) with automated manual fallback.
    - Strict lower-triangular causal autoregressive masking.
    - Output projection and residual dropout.
    """

    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        assert config.n_embd % config.n_head == 0, "n_embd must be divisible by n_head"

        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.dropout_p = config.dropout

        # Batched key, query, value projections in a single linear layer
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)

        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        # Regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # Causal mask for manual attention fallback
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.block_size, config.block_size)).view(
                1, 1, config.block_size, config.block_size
            ),
        )

        # Detect FlashAttention / SDPA support
        self.has_sdpa = hasattr(F, "scaled_dot_product_attention")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for causal self-attention.
        Input shape: (B, T, C)
        Output shape: (B, T, C)
        """
        B, T, C = x.size()

        # Compute query, key, values for all heads in batch
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # Reshape to (B, n_head, T, head_dim)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Scaled Dot-Product Attention with FlashAttention / Fallback
        if self.has_sdpa:
            # Native PyTorch SDPA automatically chooses FlashAttention, memory efficient, or math kernel
            y = F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=None,
                dropout_p=self.dropout_p if self.training else 0.0,
                is_causal=True,
            )
        else:
            # Manual fallback
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        # Reassemble all head outputs side by side: (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # Output projection
        y = self.resid_dropout(self.c_proj(y))
        return y


In [ ]:
"""
Feed-Forward Network (MLP) for Transformer Block.
Structure: Linear(n_embd, 4*n_embd) -> GELU(approx="tanh") -> Linear(4*n_embd, n_embd) -> Dropout.
"""

from __future__ import annotations

import torch
import torch.nn as nn



class MLP(nn.Module):
    """
    Position-wise Feed-Forward Network:
    Expands channel dimension by 4x, applies GELU with tanh approximation,
    projects back to n_embd, and applies residual dropout.
    """

    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


In [ ]:
"""
Transformer Block with Pre-LayerNorm Architecture.
Sequence: x -> x + Attention(LN1(x)) -> x + MLP(LN2(x))
"""

from __future__ import annotations

import torch
import torch.nn as nn



class TransformerBlock(nn.Module):
    """
    Standard Pre-LayerNorm Transformer Decoder Block:
    1. LayerNorm -> Multi-Head Causal Self-Attention -> Residual Addition
    2. LayerNorm -> Feed-Forward Network (MLP) -> Residual Addition
    """

    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN attention block with residual connection
        x = x + self.attn(self.ln_1(x))
        # Pre-LN feed-forward block with residual connection
        x = x + self.mlp(self.ln_2(x))
        return x


In [ ]:
"""
Full Decoder-Only Generative Pre-trained Transformer (GPT) Model.
Implements token & positional embeddings, stacked TransformerBlocks, final LayerNorm,
tied LM head, parameter initialization, forward pass with loss, and parameter counting.
"""

from __future__ import annotations

import math
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F



class GPT(nn.Module):
    """
    Decoder-Only Generative Pre-trained Transformer (GPT).
    Direct raw PyTorch implementation adhering strictly to GPT-2 design specifications.
    """

    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(
            dict(
                wte=nn.Embedding(config.vocab_size, config.n_embd),
                wpe=nn.Embedding(config.block_size, config.n_embd),
                drop=nn.Dropout(config.dropout),
                h=nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)]),
                ln_f=nn.LayerNorm(config.n_embd, bias=config.bias),
            )
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight tying: share weights between input token embedding and output projection
        self.transformer.wte.weight = self.lm_head.weight

        # Initialize all model weights according to GPT-2 standard
        self.apply(self._init_weights)

        # Apply special scaled initialization to the residual projections (c_proj)
        for pn, p in self.named_parameters():
            if pn.endswith("c_proj.weight"):
                torch.nn.init.normal_(
                    p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer)
                )

    def _init_weights(self, module: nn.Module) -> None:
        """Initialize weights: normal(0, 0.02) for weights, zeros for biases."""
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def get_num_params(self, non_embedding: bool = False) -> Tuple[int, int]:
        """
        Returns (total_params, trainable_params).
        If non_embedding is True, subtracts the positional embedding parameters.
        (Token embeddings are tied with lm_head, so they count once).
        """
        n_params = sum(p.numel() for p in self.parameters())
        n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
            if self.transformer.wpe.weight.requires_grad:
                n_trainable -= self.transformer.wpe.weight.numel()
        return n_params, n_trainable

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Forward pass for GPT model.
        Args:
            idx: Tensor of token indices with shape (Batch, Seq_Len)
            targets: Optional tensor of target token indices with shape (Batch, Seq_Len)
        Returns:
            logits: (Batch, Seq_Len, Vocab_Size) or (Batch, 1, Vocab_Size) during generation
            loss: scalar cross-entropy loss if targets is provided, otherwise None
        """
        # Automatically synchronize device with model parameters
        device = self.transformer.wte.weight.device
        if idx.device != device:
            idx = idx.to(device)
        if targets is not None and targets.device != device:
            targets = targets.to(device)

        b, t = idx.size()
        if t > self.config.block_size:
            idx = idx[:, -self.config.block_size:]
            t = idx.size(1)

        # Safety clamp to strictly guarantee indices are within embedding bounds
        idx = torch.clamp(idx, 0, self.config.vocab_size - 1)

        pos = torch.arange(0, t, dtype=torch.long, device=device)  # shape (t)

        # Forward token & position embeddings
        tok_emb = self.transformer.wte(idx)  # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos)  # position embeddings of shape (t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)

        # Forward through transformer blocks
        for block in self.transformer.h:
            x = block(x)

        # Final LayerNorm
        x = self.transformer.ln_f(x)

        if targets is not None:
            # If targets are provided, compute cross-entropy loss across all positions
            logits = self.lm_head(x)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )
        else:
            # Inference optimization: only project the last token logit to save memory
            logits = self.lm_head(x[:, [-1], :])  # (b, 1, vocab_size)
            loss = None

        return logits, loss

    def crop_block_size(self, block_size: int) -> None:
        """Shrink model block_size (context window) if needed."""
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(
            self.transformer.wpe.weight[:block_size]
        )
        for block in self.transformer.h:
            if hasattr(block.attn, "bias"):
                block.attn.bias = block.attn.bias[:, :, :block_size, :block_size]


# Instantiate GPT with gpt-nano preset for our character vocabulary
# You can control training duration using epochs or max_iters:
EPOCHS = 3
steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch

nano_cfg = GPTConfig.from_preset(
    "gpt-nano",
    vocab_size=char_tok.vocab_size,
    block_size=64,
    batch_size=8,
    epochs=EPOCHS,
    max_iters=total_steps,
    lr_decay_iters=total_steps,
    eval_interval=max(1, total_steps // 5),
    eval_iters=5,
    learning_rate=1e-3,
    mixed_precision="fp16" if torch.cuda.is_available() else "no",
)
print(f"Training schedule: {EPOCHS} epoch(s) x {steps_per_epoch:,} steps/epoch = {total_steps:,} total steps")

model = GPT(nano_cfg)
total_params, trainable_params = model.get_num_params()
print(f"Initialized UltraGPT: {total_params:,} total parameters ({trainable_params:,} trainable).")


## Phase 7: Training Engine (Decoupled AdamW, Cosine Warmup, AMP & Checkpointing)
We define:
1. `configure_optimizers`: Decoupled weight decay (2D weights get decay, 1D biases/norms do not).
2. `CosineWarmupScheduler`: Linear warmup followed by cosine decay.
3. `CheckpointManager`: Rolling `latest_checkpoint.pt` and versioned snapshot `checkpoint_step_{N}.pt`.
4. `Trainer`: AMP autocast, GradScaler, gradient clipping, telemetry, and loss curve plotting.


In [ ]:
"""
Optimizer Configuration with Decoupled Weight Decay.
Applies weight decay strictly to 2D tensors (weights), excluding 1D biases and LayerNorms.
"""

from __future__ import annotations

import logging
from typing import Tuple

import torch
import torch.nn as nn

logger = logging.getLogger(__name__)


def configure_optimizers(
    model: nn.Module,
    weight_decay: float = 0.1,
    learning_rate: float = 6e-4,
    betas: Tuple[float, float] = (0.9, 0.95),
    device_type: str = "cpu",
) -> torch.optim.AdamW:
    """
    Categorizes parameters into decayed and non-decayed groups:
    - 2D parameters (e.g. Linear weights, embedding weights) receive weight_decay.
    - 1D parameters (biases, LayerNorm scale/bias) receive 0.0 weight_decay.
    """
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        # Biases and 1D normalization weights do not decay
        if param.dim() >= 2:
            decay_params.append(param)
        else:
            no_decay_params.append(param)

    optim_groups = [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

    num_decay_params = sum(p.numel() for p in decay_params)
    num_nodecay_params = sum(p.numel() for p in no_decay_params)
    logger.info(
        f"Optimized parameters: {len(decay_params)} tensors with decay ({num_decay_params:,} params), "
        f"{len(no_decay_params)} tensors without decay ({num_nodecay_params:,} params)."
    )

    # Use fused AdamW kernel if running on CUDA and supported
    fused_available = "fused" in torch.optim.AdamW.__init__.__code__.co_varnames
    use_fused = fused_available and device_type == "cuda"
    extra_args = {"fused": True} if use_fused else {}

    optimizer = torch.optim.AdamW(
        optim_groups,
        lr=learning_rate,
        betas=betas,
        eps=1e-8,
        **extra_args,
    )
    return optimizer


In [ ]:
"""
Cosine Learning Rate Scheduler with Linear Warmup.
"""

from __future__ import annotations

import math
from typing import Optional
import torch



class CosineWarmupScheduler:
    """
    Computes learning rate with:
    1. Linear warmup from 0 to peak learning_rate over warmup_iters.
    2. Cosine decay from peak learning_rate to min_lr over (lr_decay_iters - warmup_iters).
    3. Flat min_lr past lr_decay_iters.
    """

    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        learning_rate: float,
        min_lr: float,
        warmup_iters: int,
        lr_decay_iters: int,
    ) -> None:
        self.optimizer = optimizer
        self.learning_rate = learning_rate
        self.min_lr = min_lr
        self.warmup_iters = warmup_iters
        self.lr_decay_iters = lr_decay_iters

    def get_lr_for_step(self, step: int) -> float:
        """Compute the learning rate for a specific global step."""
        # 1. Linear warmup
        if step < self.warmup_iters:
            return self.learning_rate * (step + 1) / (self.warmup_iters + 1e-8)
        # 2. Past decay schedule -> min_lr
        if step > self.lr_decay_iters:
            return self.min_lr
        # 3. In-between cosine decay
        decay_ratio = (step - self.warmup_iters) / max(
            1, (self.lr_decay_iters - self.warmup_iters)
        )
        coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
        return self.min_lr + coeff * (self.learning_rate - self.min_lr)

    def step(self, step: int) -> float:
        """Update optimizer learning rates to match schedule for current step."""
        lr = self.get_lr_for_step(step)
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr
        return lr


In [ ]:
"""
Checkpoint Management: Rolling latest checkpoint and periodic versioned snapshots.
Handles full state restoration for model, optimizer, scaler, and training metrics.
"""

from __future__ import annotations

import logging
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import torch
import torch.nn as nn

logger = logging.getLogger(__name__)


class CheckpointManager:
    """
    Saves and restores training checkpoints:
    - Maintains a rolling 'latest_checkpoint.pt'
    - Creates periodic versioned snapshots 'checkpoint_step_{step}.pt'
    - Preserves model state, optimizer state, scaler state, step count, and loss.
    """

    def __init__(self, checkpoint_dir: str | Path = "checkpoints") -> None:
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.latest_path = self.checkpoint_dir / "latest_checkpoint.pt"

    def save(
        self,
        model: nn.Module,
        optimizer: torch.optim.Optimizer,
        scaler: Optional[torch.amp.GradScaler],
        step: int,
        loss: float,
        config: Any,
        is_snapshot: bool = False,
    ) -> Path:
        """Serialize complete training state."""
        raw_model = model.module if hasattr(model, "module") else model
        state = {
            "step": step,
            "loss": loss,
            "model_state_dict": raw_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "config": config.to_dict() if hasattr(config, "to_dict") else config,
        }

        # Save rolling latest checkpoint
        torch.save(state, self.latest_path)

        # Save periodic versioned snapshot if requested
        if is_snapshot:
            snapshot_path = self.checkpoint_dir / f"checkpoint_step_{step}.pt"
            torch.save(state, snapshot_path)
            logger.info(f"Saved versioned checkpoint snapshot to {snapshot_path}")
            return snapshot_path

        return self.latest_path

    def resume(
        self,
        model: nn.Module,
        optimizer: Optional[torch.optim.Optimizer] = None,
        scaler: Optional[torch.amp.GradScaler] = None,
        checkpoint_path: Optional[str | Path] = None,
        device: str = "cpu",
    ) -> Tuple[int, float, Dict[str, Any]]:
        """
        Restore state from checkpoint. If path not specified, defaults to latest_checkpoint.pt.
        Returns: (step, loss, config_dict)
        """
        target_path = Path(checkpoint_path) if checkpoint_path else self.latest_path
        if not target_path.exists():
            raise FileNotFoundError(f"Checkpoint not found at {target_path}")

        logger.info(f"Loading checkpoint from {target_path} on device {device}...")
        checkpoint = torch.save if False else torch.load(target_path, map_location=device)

        raw_model = model.module if hasattr(model, "module") else model
        raw_model.load_state_dict(checkpoint["model_state_dict"])

        if optimizer is not None and "optimizer_state_dict" in checkpoint and checkpoint["optimizer_state_dict"]:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        if scaler is not None and "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"]:
            scaler.load_state_dict(checkpoint["scaler_state_dict"])

        step = checkpoint.get("step", 0)
        loss = checkpoint.get("loss", 0.0)
        config_dict = checkpoint.get("config", {})

        logger.info(f"Successfully restored checkpoint from step {step} with loss {loss:.4f}.")
        return step, loss, config_dict


In [ ]:
"""
Production Training Engine with AMP, Telemetry, and Loss Visualization.
Includes gradient clipping, tokens/sec calculation, NaN early-exit safeguard, and matplotlib plotting.
"""

from __future__ import annotations

import logging
import math
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


logger = logging.getLogger(__name__)


class Trainer:
    """
    Production-grade Training Engine for Decoder-Only GPT:
    - Native Automatic Mixed Precision (AMP) with GradScaler.
    - Decoupled AdamW optimization & Cosine Warmup LR schedule.
    - Gradient clipping and memory leak mitigation.
    - Real-time telemetry (train/val loss, lr, elapsed time, tokens/sec).
    - Post-training Matplotlib visualization of training trajectories.
    """

    def __init__(
        self,
        model: nn.Module,
        config: GPTConfig,
        train_loader: DataLoader,
        val_loader: DataLoader,
    ) -> None:
        self.model = model
        self.config = config
        self.train_loader = train_loader
        self.val_loader = val_loader

        self.device = config.device
        self.model.to(self.device)

        # Optimizer setup with decoupled weight decay
        self.optimizer = configure_optimizers(
            self.model,
            weight_decay=config.weight_decay,
            learning_rate=config.learning_rate,
            device_type=self.device,
        )

        # Cosine warmup scheduler
        self.scheduler = CosineWarmupScheduler(
            self.optimizer,
            learning_rate=config.learning_rate,
            min_lr=config.min_lr,
            warmup_iters=config.warmup_iters,
            lr_decay_iters=config.lr_decay_iters,
        )

        # Mixed precision setup
        # For CUDA: 'fp16' or 'bf16'; for CPU/MPS: default to float32
        self.device_type = "cuda" if "cuda" in self.device else "cpu"
        self.pt_dtype = {
            "fp16": torch.float16,
            "bf16": torch.bfloat16,
            "no": torch.float32,
        }.get(config.mixed_precision, torch.float32)

        self.use_amp = self.device_type == "cuda" and config.mixed_precision in ("fp16", "bf16")
        self.scaler = torch.amp.GradScaler("cuda", enabled=(self.use_amp and config.mixed_precision == "fp16"))

        # Checkpoint manager
        self.checkpoint_mgr = CheckpointManager(config.checkpoint_dir)

        # History tracking for telemetry and plotting
        self.history: Dict[str, List[float]] = {
            "steps": [],
            "train_loss": [],
            "val_steps": [],
            "val_loss": [],
            "lr": [],
            "tokens_per_sec": [],
        }

    @torch.no_grad()
    def estimate_loss(self) -> Tuple[float, float]:
        """Estimate mean loss on train and validation sets over eval_iters batches."""
        self.model.eval()
        losses = {"train": 0.0, "val": 0.0}

        for split, loader in [("train", self.train_loader), ("val", self.val_loader)]:
            total_loss = 0.0
            count = 0
            for i, (x, y) in enumerate(loader):
                if i >= self.config.eval_iters:
                    break
                x, y = x.to(self.device, non_blocking=True), y.to(self.device, non_blocking=True)
                with torch.amp.autocast(device_type=self.device_type, dtype=self.pt_dtype, enabled=self.use_amp):
                    _, loss = self.model(x, y)
                total_loss += loss.item()
                count += 1
            losses[split] = total_loss / max(1, count)

        self.model.train()
        return losses["train"], losses["val"]

    def train(self, resume_from_checkpoint: Optional[bool] = None) -> Dict[str, List[float]]:
        """
        Main training loop with telemetry logging, checkpointing, and safeguards.
        - If resume_from_checkpoint is True or None (auto), and latest_checkpoint.pt exists,
          it automatically resumes training from the latest weights, step, and optimizer state!
        - If resume_from_checkpoint is False, starts afresh from step 0.
        """
        start_step = 0
        should_resume = (
            resume_from_checkpoint is True
            or (resume_from_checkpoint is None and self.checkpoint_mgr.latest_path.exists())
        )

        if should_resume and self.checkpoint_mgr.latest_path.exists():
            print(f"\n[Auto-Resume] Found existing checkpoint at: {self.checkpoint_mgr.latest_path}")
            start_step, last_loss, _ = self.checkpoint_mgr.resume(
                self.model, self.optimizer, self.scaler, device=self.device
            )
            print(f"  -> Successfully restored weights at Step {start_step:,} (Loss: {last_loss:.4f})")
            if self.config.max_iters <= start_step:
                added = 1000
                self.config.max_iters = start_step + added
                self.config.lr_decay_iters = self.config.max_iters
                print(f"  -> Target step {start_step:,} already achieved. Extending training to {self.config.max_iters:,} steps (+{added:,}).")
        elif resume_from_checkpoint is True:
            raise FileNotFoundError(f"Cannot resume: No checkpoint found at {self.checkpoint_mgr.latest_path}")

        self.model.train()
        data_iter = iter(self.train_loader)
        tokens_per_batch = self.config.batch_size * self.config.block_size
        t0 = time.time()
        start_time = time.time()

        print(f"\n--- Launching UltraGPT Training on {self.device.upper()} ---")
        print(f"Max Iters: {self.config.max_iters} | Batch Size: {self.config.batch_size} | Context: {self.config.block_size}")
        print(f"Mixed Precision: {self.config.mixed_precision} | AMP Enabled: {self.use_amp}\n")

        # Throttled progress bar: updates at most once per second to prevent VS Code Jupyter buffer crashes
        pbar = tqdm(
            range(start_step, self.config.max_iters),
            desc="Training UltraGPT",
            unit="step",
            mininterval=1.0,
            dynamic_ncols=True,
        )

        ema_loss = None
        last_val_loss = None

        try:
            for step in pbar:
                step_start = time.time()

                # Dynamic LR step
                lr = self.scheduler.step(step)

                # Fetch microbatch with cycling
                try:
                    x, y = next(data_iter)
                except StopIteration:
                    data_iter = iter(self.train_loader)
                    x, y = next(data_iter)

                x, y = x.to(self.device, non_blocking=True), y.to(self.device, non_blocking=True)

                # Forward pass under AMP autocast
                with torch.amp.autocast(device_type=self.device_type, dtype=self.pt_dtype, enabled=self.use_amp):
                    _, loss = self.model(x, y)

                loss_val = loss.item()

                # Early-exit safeguard for numerical instability / NaN
                if math.isnan(loss_val) or math.isinf(loss_val):
                    raise FloatingPointError(f"Training aborted: Loss exploded to {loss_val} at step {step}!")

                # Backward pass with GradScaler
                self.optimizer.zero_grad(set_to_none=True)
                self.scaler.scale(loss).backward()

                # Gradient clipping to stabilize Transformer gradients
                if self.config.grad_clip > 0.0:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.grad_clip)

                self.scaler.step(self.optimizer)
                self.scaler.update()

                # Calculate real-time EMA loss and instantaneous step throughput
                ema_loss = loss_val if ema_loss is None else (0.95 * ema_loss + 0.05 * loss_val)
                step_dt = max(time.time() - step_start, 1e-5)
                step_tok_s = tokens_per_batch / step_dt

                # Update live real-time progress bar (throttled to 1 update/sec)
                pbar.set_postfix(
                    {
                        "step_loss": f"{loss_val:.4f}",
                        "ema_loss": f"{ema_loss:.4f}",
                        "val_loss": f"{last_val_loss:.4f}" if last_val_loss is not None else "n/a",
                        "lr": f"{lr:.2e}",
                        "tok/s": f"{step_tok_s:,.0f}",
                    }
                )

                # Periodic Evaluation & Telemetry
                if (step + 1) % self.config.eval_interval == 0 or step == self.config.max_iters - 1:
                    t1 = time.time()
                    dt = t1 - t0
                    tokens_processed = tokens_per_batch * self.config.eval_interval
                    tokens_per_sec = tokens_processed / max(dt, 1e-6)
                    t0 = time.time()

                    train_eval_loss, val_loss = self.estimate_loss()
                    last_val_loss = val_loss

                    self.history["steps"].append(step + 1)
                    self.history["train_loss"].append(train_eval_loss)
                    self.history["val_steps"].append(step + 1)
                    self.history["val_loss"].append(val_loss)
                    self.history["lr"].append(lr)
                    self.history["tokens_per_sec"].append(tokens_per_sec)

                    elapsed_str = time.strftime("%H:%M:%S", time.gmtime(time.time() - start_time))
                    pbar.write(
                        f"[Step {step + 1:5d}/{self.config.max_iters}] | "
                        f"Train: {train_eval_loss:.4f} | "
                        f"Val: {val_loss:.4f} | "
                        f"LR: {lr:.2e} | "
                        f"Avg Speed: {tokens_per_sec:7,.0f} tok/s | "
                        f"Elapsed: {elapsed_str}"
                    )

                    # Periodic snapshot checkpoint
                    is_snapshot = ((step + 1) % self.config.save_checkpoint_interval == 0)
                    self.checkpoint_mgr.save(
                        self.model,
                        self.optimizer,
                        self.scaler,
                        step=step + 1,
                        loss=val_loss,
                        config=self.config,
                        is_snapshot=is_snapshot,
                    )

                # Prevent CUDA memory fragmentation leak
                if self.device_type == "cuda" and (step + 1) % 500 == 0:
                    torch.cuda.empty_cache()
        finally:
            pbar.close()
            del data_iter
            self.optimizer.zero_grad(set_to_none=True)
            if self.device_type == "cuda":
                torch.cuda.empty_cache()

        print(f"\nTraining completed in {time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))}.")
        return self.history

    def plot_metrics(self, save_path: Optional[str | Path] = "training_metrics.png") -> plt.Figure:
        """Plot loss curves (train vs. validation) and learning rate trajectory."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # Loss Curve
        ax1.plot(self.history["steps"], self.history["train_loss"], label="Train Loss", color="#1f77b4", lw=2)
        ax1.plot(self.history["val_steps"], self.history["val_loss"], label="Val Loss", color="#ff7f0e", lw=2, linestyle="--")
        ax1.set_title("Training & Validation Loss Curve", fontsize=13, fontweight="bold")
        ax1.set_xlabel("Iteration Step", fontsize=11)
        ax1.set_ylabel("Cross-Entropy Loss", fontsize=11)
        ax1.grid(True, linestyle=":", alpha=0.6)
        ax1.legend(fontsize=11)

        # Learning Rate Schedule
        ax2.plot(self.history["steps"], self.history["lr"], label="Learning Rate", color="#2ca02c", lw=2)
        ax2.set_title("Cosine Warmup Learning Rate Trajectory", fontsize=13, fontweight="bold")
        ax2.set_xlabel("Iteration Step", fontsize=11)
        ax2.set_ylabel("Learning Rate", fontsize=11)
        ax2.grid(True, linestyle=":", alpha=0.6)
        ax2.legend(fontsize=11)

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300)
            print(f"Metrics plot saved to {save_path}")
        return fig


## Phase 8: Model Training Execution & Resuming Mechanism
We train our UltraGPT model with live real-time telemetry. If `checkpoints/latest_checkpoint.pt` exists, **it automatically resumes from the saved step and trained weights** instead of starting from scratch!


In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    config=nano_cfg,
    train_loader=train_loader,
    val_loader=val_loader,
)

# Check for existing checkpoint
ckpt_file = Path("checkpoints/latest_checkpoint.pt")
resume_mode = ckpt_file.exists()

if resume_mode:
    print(f"[*] Found checkpoint at {ckpt_file}. RESUMING training from saved weights...")
else:
    print("[*] No checkpoint found. Starting training from scratch...")

# trainer.train() automatically restores model weights, optimizer buffers, and step count
training_history = trainer.train(resume_from_checkpoint=resume_mode)


## Phase 9: Telemetry Visualizations
We generate and display the cross-entropy loss curve (train vs. validation) and the learning rate trajectory.


In [ ]:
import matplotlib.pyplot as plt

fig = trainer.plot_metrics(save_path="training_metrics.png")
plt.show()


## Phase 10: Context-Aware Conversational Bot
We implement:
- `sample_next_token`: Temperature, Top-K, Top-P (nucleus), and repetition penalty.
- `ConversationBot`: Manages multi-turn history with sliding window truncation respecting `block_size`, and yields real-time streaming tokens.


In [ ]:
"""
Token Sampling Strategies for Autoregressive Generation.
Implements Temperature, Top-K, Nucleus (Top-P) Filtering, and Repetition Penalties.
"""

from __future__ import annotations

from typing import List, Optional, Set
import torch
import torch.nn.functional as F


def sample_next_token(
    logits: torch.Tensor,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
    repetition_penalty: float = 1.0,
    generated_tokens: Optional[List[int]] = None,
) -> int:
    """
    Selects the next token from unnormalized logits:
    1. Applies repetition penalty to previously emitted tokens.
    2. Temperature scaling.
    3. Top-K filtering.
    4. Top-P (nucleus) cumulative mass filtering.
    5. Categorical sampling via torch.multinomial (or argmax if temperature=0).
    """
    # Flatten to 1D vector (vocab_size,)
    logits = logits.squeeze()

    # 1. Repetition Penalty
    if repetition_penalty != 1.0 and generated_tokens:
        seen_tokens = set(generated_tokens)
        for token_id in seen_tokens:
            if token_id < logits.size(0):
                val = logits[token_id].item()
                if val > 0:
                    logits[token_id] /= repetition_penalty
                else:
                    logits[token_id] *= repetition_penalty

    # 2. Greedy decoding when temperature is effectively zero
    if temperature <= 1e-4:
        return int(torch.argmax(logits).item())

    # Temperature Scaling
    logits = logits / temperature

    # 3. Top-K Filtering
    if top_k is not None and top_k > 0:
        top_k = min(max(top_k, 1), logits.size(-1))
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[-1]] = float("-inf")

    # 4. Top-P (Nucleus) Filtering
    if top_p is not None and 0.0 < top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        # Remove tokens with cumulative probability above threshold
        sorted_indices_to_remove = cumulative_probs > top_p
        # Shift the indices to the right to keep also the first token above threshold
        sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
        sorted_indices_to_remove[0] = False

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[indices_to_remove] = float("-inf")

    # 5. Categorical Sampling
    probs = F.softmax(logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    return int(next_token.item())


In [ ]:
"""
Context-Aware Conversational Bot with Sliding Window Memory and Streaming Generation.
"""

from __future__ import annotations

from typing import Dict, Generator, List, Optional, Set
import torch
import torch.nn as nn



class ConversationBot:
    """
    Stateful conversational agent wrapper:
    - Manages multi-turn conversation history.
    - Context memory manager: slides/truncates oldest history to strictly fit within model block_size.
    - Supports streaming token-by-token generation or synchronous execution.
    - Configurable sampling (temp, top-k, top-p, repetition penalty, stop tokens).
    """

    def __init__(
        self,
        model: nn.Module,
        tokenizer: BaseTokenizer,
        system_prompt: str = "You are UltraGPT, a helpful and precise AI assistant.",
        device: str = "cpu",
    ) -> None:
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
        self.device = device
        self.block_size = getattr(model.config, "block_size", 1024)

        self.messages: List[Dict[str, str]] = []
        self.reset()

    def reset(self) -> None:
        """Clear conversational history, retaining system prompt."""
        self.messages = [{"role": "system", "content": self.system_prompt}]

    def _truncate_history(self, new_user_msg: str, max_new_tokens: int) -> str:
        """
        Dynamically trims conversation history so prompt tokens + max_new_tokens <= block_size.
        Always preserves system prompt (messages[0]).
        """
        candidate_messages = list(self.messages) + [{"role": "user", "content": new_user_msg}]
        max_prompt_len = self.block_size - max_new_tokens - 8  # headroom for formatting

        while len(candidate_messages) > 2:
            prompt_str = DialogueFormatter.format_conversation(candidate_messages, add_generation_prompt=True)
            tokens = self.tokenizer.encode(prompt_str)
            if len(tokens) <= max_prompt_len:
                break
            # Remove oldest message after system prompt
            candidate_messages.pop(1)

        prompt_str = DialogueFormatter.format_conversation(candidate_messages, add_generation_prompt=True)
        return prompt_str

    @torch.no_grad()
    def stream_generate(
        self,
        user_message: str,
        max_tokens: int = 256,
        temperature: float = 0.8,
        top_k: Optional[int] = 50,
        top_p: Optional[float] = 0.9,
        repetition_penalty: float = 1.1,
        stop_tokens: Optional[List[str]] = None,
    ) -> Generator[str, None, str]:
        """
        Stream generation token by token:
        Yields newly emitted text fragments as they are generated.
        Returns the full assistant response at end.
        """
        self.model.eval()

        prompt_text = self._truncate_history(user_message, max_new_tokens=max_tokens)
        input_ids = self.tokenizer.encode(prompt_text)
        current_tokens = list(input_ids)
        generated_token_ids: List[int] = []

        # Default stop tokens
        stops: Set[str] = {DialogueFormatter.IM_END, "<|endoftext|>"}
        if stop_tokens:
            stops.update(stop_tokens)

        # Stop token IDs
        stop_ids = {self.tokenizer.eos_token_id}
        if hasattr(self.tokenizer, "im_end_token_id"):
            stop_ids.add(self.tokenizer.im_end_token_id)

        accumulated_text = ""

        for _ in range(max_tokens):
            # Take last block_size tokens and clamp to valid vocabulary range
            context_window = current_tokens[-self.block_size :]
            vocab_size = getattr(self.model.config, "vocab_size", 50257)
            context_window = [min(max(0, tid), vocab_size - 1) for tid in context_window]
            model_device = next(self.model.parameters()).device
            x = torch.tensor([context_window], dtype=torch.long, device=model_device)

            logits, _ = self.model(x)  # (1, 1, vocab_size)
            next_logits = logits[0, -1, :]

            next_token = sample_next_token(
                logits=next_logits,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                generated_tokens=generated_token_ids,
            )

            # Check stop token IDs
            if next_token in stop_ids:
                break

            current_tokens.append(next_token)
            generated_token_ids.append(next_token)

            # Decode newest token
            piece = self.tokenizer.decode([next_token])
            accumulated_text += piece

            # Check textual stop triggers
            stop_triggered = False
            for st in stops:
                if st in accumulated_text:
                    accumulated_text = accumulated_text.split(st)[0]
                    stop_triggered = True
                    break

            if stop_triggered:
                break

            yield piece

        clean_response = accumulated_text.strip()
        # Record turn in persistent history
        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": clean_response})

        return clean_response

    def generate(
        self,
        user_message: str,
        max_tokens: int = 256,
        temperature: float = 0.8,
        top_k: Optional[int] = 50,
        top_p: Optional[float] = 0.9,
        repetition_penalty: float = 1.1,
        stop_tokens: Optional[List[str]] = None,
    ) -> str:
        """Synchronously generate complete response for user message."""
        gen = self.stream_generate(
            user_message=user_message,
            max_tokens=max_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            stop_tokens=stop_tokens,
        )
        full_text = ""
        for piece in gen:
            full_text += piece
        return full_text.strip()


# Initialize interactive conversational bot with trained model
# If a saved checkpoint exists, restore the exact trained weights & vocab
ckpt_path = Path("checkpoints/latest_checkpoint.pt")
vocab_path = Path("checkpoints/char_vocab.json")

if ckpt_path.exists():
    print(f"Loading trained weights from {ckpt_path}...")
    ckpt = torch.load(ckpt_path, map_location=device)
    active_cfg = GPTConfig(**ckpt["config"])
    active_model = GPT(active_cfg)
    active_model.load_state_dict(ckpt["model_state_dict"])
    active_model.to(device)
    active_model.eval()
    if vocab_path.exists():
        active_tok = CharTokenizer.load(vocab_path)
    else:
        active_tok = char_tok
    print(f"Successfully restored trained checkpoint: Step {ckpt.get('step', 0):,}, Loss {ckpt.get('loss', 0):.4f}, Vocab {active_cfg.vocab_size}")
else:
    active_model = model
    active_tok = char_tok
    active_cfg = nano_cfg

bot = ConversationBot(
    model=active_model,
    tokenizer=active_tok,
    system_prompt="You are UltraGPT, an expert AI assistant.",
    device=active_cfg.device,
)

print("\nPrompt: 'What is deep learning?'")
print("UltraGPT > ", end="", flush=True)
for piece in bot.stream_generate("What is deep learning?", max_tokens=60, temperature=0.7):
    print(piece, end="", flush=True)
print()


## Phase 11: OpenAI-Compatible FastAPI Serving Layer & Live Verification
We load the standalone FastAPI server and verify the endpoints using an in-process HTTP test client.


In [ ]:
"""
OpenAI-Compatible FastAPI Serving Layer for UltraGPT.
Endpoints:
- POST /v1/chat/completions (Supports both JSON response and Server-Sent Events streaming)
- GET  /v1/models (Metadata & active architecture preset)
- GET  /health (Health check & device status)
"""

from __future__ import annotations

import json
import time
import uuid
from contextlib import asynccontextmanager
from pathlib import Path
from typing import AsyncGenerator, Dict, List, Literal, Optional, Union

import torch
from fastapi import FastAPI, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field



# =========================================================
# Pydantic Schemas strictly conforming to OpenAI API Spec
# =========================================================

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"] = "user"
    content: str


class ChatCompletionRequest(BaseModel):
    model: str = "ultra-gpt"
    messages: List[ChatMessage]
    temperature: Optional[float] = Field(default=0.8, ge=0.0, le=2.0)
    top_p: Optional[float] = Field(default=0.9, ge=0.0, le=1.0)
    top_k: Optional[int] = Field(default=50, ge=0)
    repetition_penalty: Optional[float] = Field(default=1.1, ge=1.0, le=2.0)
    max_tokens: Optional[int] = Field(default=256, ge=1, le=4096)
    stream: Optional[bool] = False
    stop: Optional[Union[str, List[str]]] = None


class UsageInfo(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int


class ChatCompletionResponseChoice(BaseModel):
    index: int
    message: ChatMessage
    finish_reason: Literal["stop", "length"]


class ChatCompletionResponse(BaseModel):
    id: str
    object: str = "chat.completion"
    created: int
    model: str
    choices: List[ChatCompletionResponseChoice]
    usage: UsageInfo


class ChatCompletionChunkDelta(BaseModel):
    role: Optional[str] = None
    content: Optional[str] = None


class ChatCompletionChunkChoice(BaseModel):
    index: int
    delta: ChatCompletionChunkDelta
    finish_reason: Optional[Literal["stop", "length"]] = None


class ChatCompletionChunk(BaseModel):
    id: str
    object: str = "chat.completion.chunk"
    created: int
    model: str
    choices: List[ChatCompletionChunkChoice]


# =========================================================
# Global Server State & Lifespan Loader
# =========================================================

class ServerState:
    model: Optional[GPT] = None
    tokenizer: Optional[Union[BPETokenizer, CharTokenizer]] = None
    config: Optional[GPTConfig] = None
    device: str = "cpu"


state = ServerState()


def load_model_and_tokenizer(
    checkpoint_path: Optional[str] = None,
    preset: str = "gpt-nano",
    use_bpe: bool = False,
) -> None:
    """Load model and tokenizer into global server state."""
    # Determine config
    if checkpoint_path and Path(checkpoint_path).exists():
        ckpt = torch.load(checkpoint_path, map_location="cpu")
        cfg_dict = ckpt.get("config", {})
        config = GPTConfig(**cfg_dict)
    else:
        config = GPTConfig.from_preset(preset)

    device = config.device
    model = GPT(config)

    if checkpoint_path and Path(checkpoint_path).exists():
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])

    model.to(device)
    model.eval()

    # Determine tokenizer
    if use_bpe:
        try:
            tokenizer = BPETokenizer()
        except Exception:
            tokenizer = CharTokenizer()
    else:
        # Default self-contained CharTokenizer
        tokenizer = CharTokenizer()
        # Seed default printable chars
        import string
        tokenizer.build_vocab(string.ascii_letters + string.digits + string.punctuation + " \n\t")

    state.model = model
    state.config = config
    state.tokenizer = tokenizer
    state.device = device


@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup: Initialize default model if not already present
    if state.model is None:
        load_model_and_tokenizer(preset="gpt-nano")
    yield
    # Shutdown logic
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


app = FastAPI(
    title="UltraGPT OpenAI-Compatible API",
    description="Production-grade OpenAI-compatible Serving API for Decoder-Only GPT",
    version="1.0.0",
    lifespan=lifespan,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# =========================================================
# Server Endpoints
# =========================================================

@app.get("/health", tags=["Health"])
async def health_check() -> Dict[str, Any]:
    """Health status and device telemetry."""
    if state.model is None or state.config is None:
        raise HTTPException(status_code=503, detail="Model is not initialized")

    total_params, trainable_params = state.model.get_num_params()
    return {
        "status": "healthy",
        "device": state.device,
        "cuda_available": torch.cuda.is_available(),
        "preset": state.config.preset_name or "custom",
        "vocab_size": state.config.vocab_size,
        "block_size": state.config.block_size,
        "parameters": {
            "total": total_params,
            "trainable": trainable_params,
        },
    }


@app.get("/v1/models", tags=["OpenAI"])
async def list_models() -> Dict[str, Any]:
    """OpenAI standard models list endpoint."""
    total_params = state.model.get_num_params()[0] if state.model else 0
    return {
        "object": "list",
        "data": [
            {
                "id": "ultra-gpt",
                "object": "model",
                "created": int(time.time()),
                "owned_by": "ultra-gpt-team",
                "permission": [],
                "root": "ultra-gpt",
                "parent": None,
                "metadata": {
                    "preset": state.config.preset_name if state.config else "gpt-nano",
                    "total_parameters": total_params,
                },
            }
        ],
    }


async def stream_chat_completions(
    req: ChatCompletionRequest,
    prompt_ids: List[int],
    req_id: str,
    created_time: int,
) -> AsyncGenerator[str, None]:
    """Generates Server-Sent Events (SSE) compatible with OpenAI streaming client."""
    # Yield initial chunk with role
    init_chunk = ChatCompletionChunk(
        id=req_id,
        created=created_time,
        model=req.model,
        choices=[
            ChatCompletionChunkChoice(
                index=0,
                delta=ChatCompletionChunkDelta(role="assistant"),
                finish_reason=None,
            )
        ],
    )
    yield f"data: {init_chunk.model_dump_json()}\n\n"

    current_tokens = list(prompt_ids)
    generated_tokens: List[int] = []
    block_size = state.config.block_size
    stop_ids = {state.tokenizer.eos_token_id}
    if hasattr(state.tokenizer, "im_end_token_id"):
        stop_ids.add(state.tokenizer.im_end_token_id)

    max_toks = req.max_tokens or 256
    accumulated_str = ""

    for step in range(max_toks):
        context = current_tokens[-block_size:]
        x = torch.tensor([context], dtype=torch.long, device=state.device)

        with torch.no_grad():
            logits, _ = state.model(x)

        next_logits = logits[0, -1, :]
        next_token = sample_next_token(
            logits=next_logits,
            temperature=req.temperature or 0.8,
            top_k=req.top_k,
            top_p=req.top_p,
            repetition_penalty=req.repetition_penalty or 1.1,
            generated_tokens=generated_tokens,
        )

        if next_token in stop_ids:
            break

        current_tokens.append(next_token)
        generated_tokens.append(next_token)

        delta_piece = state.tokenizer.decode([next_token])
        accumulated_str += delta_piece

        # Check textual stop sequence
        if req.stop:
            stops = [req.stop] if isinstance(req.stop, str) else req.stop
            if any(st in accumulated_str for st in stops):
                break

        chunk = ChatCompletionChunk(
            id=req_id,
            created=created_time,
            model=req.model,
            choices=[
                ChatCompletionChunkChoice(
                    index=0,
                    delta=ChatCompletionChunkDelta(content=delta_piece),
                    finish_reason=None,
                )
            ],
        )
        yield f"data: {chunk.model_dump_json()}\n\n"

    # Final termination chunk
    final_chunk = ChatCompletionChunk(
        id=req_id,
        created=created_time,
        model=req.model,
        choices=[
            ChatCompletionChunkChoice(
                index=0,
                delta=ChatCompletionChunkDelta(),
                finish_reason="stop",
            )
        ],
    )
    yield f"data: {final_chunk.model_dump_json()}\n\n"
    yield "data: [DONE]\n\n"


@app.post("/v1/chat/completions", tags=["OpenAI"])
async def create_chat_completion(request: ChatCompletionRequest):
    """
    OpenAI-compatible Chat Completions endpoint.
    Processes multi-turn messages, applies ChatML dialogue formatting,
    and returns either standard JSON response or SSE token streaming.
    """
    if state.model is None or state.tokenizer is None or state.config is None:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="UltraGPT model is not initialized",
        )

    # Format multi-turn messages into model ChatML template
    messages_payload = [{"role": m.role, "content": m.content} for m in request.messages]
    formatted_prompt = DialogueFormatter.format_conversation(messages_payload, add_generation_prompt=True)
    prompt_tokens = state.tokenizer.encode(formatted_prompt)

    req_id = f"chatcmpl-{uuid.uuid4().hex[:12]}"
    created_time = int(time.time())

    # Handle streaming request
    if request.stream:
        return StreamingResponse(
            stream_chat_completions(request, prompt_tokens, req_id, created_time),
            media_type="text/event-stream",
            headers={
                "Cache-Control": "no-cache",
                "Connection": "keep-alive",
            },
        )

    # Synchronous full generation
    current_tokens = list(prompt_tokens)
    generated_tokens: List[int] = []
    block_size = state.config.block_size
    stop_ids = {state.tokenizer.eos_token_id}
    if hasattr(state.tokenizer, "im_end_token_id"):
        stop_ids.add(state.tokenizer.im_end_token_id)

    max_toks = request.max_tokens or 256
    accumulated_str = ""

    for _ in range(max_toks):
        context = current_tokens[-block_size:]
        x = torch.tensor([context], dtype=torch.long, device=state.device)

        with torch.no_grad():
            logits, _ = state.model(x)

        next_logits = logits[0, -1, :]
        next_token = sample_next_token(
            logits=next_logits,
            temperature=request.temperature or 0.8,
            top_k=request.top_k,
            top_p=request.top_p,
            repetition_penalty=request.repetition_penalty or 1.1,
            generated_tokens=generated_tokens,
        )

        if next_token in stop_ids:
            break

        current_tokens.append(next_token)
        generated_tokens.append(next_token)

        piece = state.tokenizer.decode([next_token])
        accumulated_str += piece

        if request.stop:
            stops = [request.stop] if isinstance(request.stop, str) else request.stop
            if any(st in accumulated_str for st in stops):
                for st in stops:
                    accumulated_str = accumulated_str.split(st)[0]
                break

    clean_content = accumulated_str.strip()
    finish_reason = "stop" if len(generated_tokens) < max_toks else "length"

    return ChatCompletionResponse(
        id=req_id,
        created=created_time,
        model=request.model,
        choices=[
            ChatCompletionResponseChoice(
                index=0,
                message=ChatMessage(role="assistant", content=clean_content),
                finish_reason=finish_reason,
            )
        ],
        usage=UsageInfo(
            prompt_tokens=len(prompt_tokens),
            completion_tokens=len(generated_tokens),
            total_tokens=len(prompt_tokens) + len(generated_tokens),
        ),
    )


# In-Process Live Verification of FastAPI Server
from starlette.testclient import TestClient

# Load trained model state into server
load_model_and_tokenizer(checkpoint_path="checkpoints/latest_checkpoint.pt", preset="gpt-nano")
client = TestClient(app)

# 1. Test /health
res_health = client.get("/health")
print("GET /health ->", res_health.json())

# 2. Test /v1/models
res_models = client.get("/v1/models")
print("GET /v1/models ->", res_models.json())

# 3. Test /v1/chat/completions (OpenAI Format)
res_chat = client.post(
    "/v1/chat/completions",
    json={
        "model": "ultra-gpt",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Explain transformers."}
        ],
        "temperature": 0.7,
        "max_tokens": 30
    }
)
print("POST /v1/chat/completions response:")
print(json.dumps(res_chat.json(), indent=2))
print("\n>>> End-to-End Verification Complete: All 11 Phases Executed Successfully! <<<")
